# CodeGuard AI

Multi-agent static and AI-assisted code review for Python. Three independent agents, each
responsible for a single analysis domain, produce findings that a fourth agent aggregates into a
weighted score and a single report.

| | |
|---|---|
| Language | Python |
| Runtime | Google Colab, T4 GPU recommended |
| Model | `deepseek-ai/deepseek-coder-1.3b-instruct` |
| UI | Gradio |

## Changelog

### Fixed
- SQL Injection rule matched only the string literal passed directly to `execute()`. Query
  strings assembled in a variable before the call were not flagged. The rule now scans for
  SQL-keyword concatenation anywhere in the source, independent of call site.
- Hardcoded Secret rule required an exact match on `password`, `secret`, `api_key`, or `token` as
  a standalone identifier. Compound names such as `ADMIN_PASSWORD` were not matched due to the
  underscore breaking the word boundary. Pattern updated to match the keyword as a substring of
  the identifier.
- Performance Agent prompt had no constraint against conflating a syntax rewrite with a change in
  algorithmic complexity. A static rule and a prompt constraint were added; see Section 8.

### Changed
- Scoring penalties increased (Critical 30, High 18, Medium 10, Low 4) so that code with a single
  critical finding cannot score above 70 for that domain.
- Agent prompts constrained to report only evidenced findings, with an explicit no-finding output
  format, to reduce fabricated issues on clean code.


## Architecture

```
                        Source (Python)
                              |
                     CodeReviewOrchestrator
                              |
        --------------------------------------------
        |                    |                      |
   SecurityAgent      PerformanceAgent        QualityAgent
   (regex scan          (AST scan             (AST scan
    + LLM)                + LLM)                + LLM)
        |                    |                      |
        --------------------------------------------
                              |
                         ReportAgent
              (weighted score, summary, patch)
                              |
                         Gradio UI
```

Each agent owns exactly one domain and does not comment on the others. All three share a single
loaded model instance (`AIEngine`), so the model is initialized once per session. Findings are
produced by deterministic static checks first; the model is used only to explain and, where
directly evidenced, extend those findings. Scores are computed in Python from finding severity and
are not influenced by model output.


## Dependencies

In [ ]:
!pip install -q transformers accelerate torch pandas gradio


## Configuration

Project identifiers, the inline SVG mark, dashboard CSS, and the rule sets injected into every
agent prompt. Keeping these in one cell avoids duplicating style and prompt-constraint strings
across the agent definitions below.

In [ ]:
import re
import ast
import datetime
from collections import Counter

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from IPython.display import display, Markdown, HTML

PROJECT_NAME = "CodeGuard AI"
MODEL_NAME = "deepseek-ai/deepseek-coder-1.3b-instruct"

# Inline SVG mark. Avoids bundling an external image asset with the notebook.
LOGO_SVG = """<svg width="20" height="20" viewBox="0 0 64 64" xmlns="http://www.w3.org/2000/svg">
  <rect x="6" y="6" width="52" height="52" rx="10" fill="none" stroke="#58a6ff" stroke-width="3"/>
  <text x="32" y="41" font-family="ui-monospace, monospace" font-size="20" fill="#58a6ff" text-anchor="middle">&lt;/&gt;</text>
</svg>"""

CUSTOM_CSS = """
.cg-header { display:flex; align-items:center; gap:10px; padding:12px 16px; background:#161b22;
             border:1px solid #30363d; border-radius:6px; margin-bottom:8px; }
.cg-header .cg-title { font-size:14px; font-weight:600; color:#c9d1d9; letter-spacing:-0.01em; }
.cg-header .cg-subtitle { font-size:12px; color:#8b949e; margin-top:1px; }
.cg-status { margin-left:auto; font-size:12px; color:#8b949e; display:flex; align-items:center; gap:6px; }
.cg-dot { width:6px; height:6px; border-radius:50%; display:inline-block; }
.cg-dot.ready { background:#3fb950; }
.cg-dot.off { background:#f85149; }
.cg-flow { background:#161b22; border:1px solid #30363d; border-radius:6px; padding:8px 14px;
           color:#8b949e; font-size:12px; font-family:ui-monospace, monospace; margin-bottom:8px; }
.cg-overall { background:#161b22; border:1px solid #30363d; border-radius:6px; padding:14px 16px;
              display:flex; justify-content:space-between; align-items:baseline; }
.cg-overall .label { font-size:11px; color:#8b949e; text-transform:uppercase; letter-spacing:0.04em; }
.cg-overall .value { font-size:26px; font-weight:600; color:#c9d1d9; font-family:ui-monospace, monospace; }
.cg-cards { display:flex; gap:8px; margin-top:8px; }
.cg-card { flex:1; background:#161b22; border:1px solid #30363d; border-left:2px solid #30363d;
           border-radius:6px; padding:10px 12px; }
.cg-card .label { font-size:11px; color:#8b949e; text-transform:uppercase; letter-spacing:0.04em; }
.cg-card .value { font-size:18px; font-weight:600; font-family:ui-monospace, monospace; margin-top:4px; color:#c9d1d9; }
.cg-card.security { border-left-color:#f85149; }
.cg-card.performance { border-left-color:#d29922; }
.cg-card.quality { border-left-color:#3fb950; }
.cg-issues { background:#161b22; border:1px solid #30363d; border-radius:6px; padding:8px 14px;
             margin-top:8px; font-size:12px; color:#8b949e; font-family:ui-monospace, monospace; }
"""

# Constraints injected into every agent prompt to reduce fabricated findings.
COMMON_AGENT_RULES = """Follow these constraints:
1. Report only issues supported directly by the code or by the findings listed below.
2. Do not invent issues that are not present.
3. If nothing significant is found beyond the listed findings, respond with exactly: "No significant issue detected."
4. Do not exaggerate severity.
5. Be concise; do not pad the response with generic advice.
6. Stay within your assigned domain.
7. Preserve the original functionality of the code when suggesting a fix.
8. Separate observed facts from recommendations."""

PERFORMANCE_RULES = """9. A syntax rewrite (list comprehension, map/filter) does not change algorithmic complexity.
10. Only an algorithmic change (for example, a set or dict lookup replacing a linear scan) reduces complexity.
11. State explicitly whether a suggestion affects (a) algorithmic complexity, (b) constant-factor runtime, (c) memory, or (d) readability."""


## Model Loading

`AIEngine` wraps model initialization and inference behind a single interface shared by all three domain agents. If loading fails, `generate()` returns a fixed message and the pipeline continues using static findings only.

In [ ]:
class AIEngine:
    """Loads the model once; shared across Security, Performance, and Quality agents."""

    def __init__(self, model_name: str = MODEL_NAME):
        self.model_name = model_name
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        self.tokenizer = None
        self.model = None
        self.load_error = None
        try:
            self.tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
            self.model = AutoModelForCausalLM.from_pretrained(
                model_name, trust_remote_code=True,
                torch_dtype=torch.float16 if self.device == "cuda" else torch.float32,
            ).to(self.device)
            self.model.eval()
        except Exception as e:
            self.load_error = str(e)

    @property
    def is_ready(self) -> bool:
        return self.model is not None

    def generate(self, prompt: str, max_new_tokens: int = 500) -> str:
        if not self.is_ready:
            return f"Model unavailable ({self.load_error}). Static findings only."
        try:
            if getattr(self.tokenizer, "chat_template", None):
                messages = [{"role": "user", "content": prompt}]
                text = self.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
            else:
                text = prompt
            inputs = self.tokenizer(text, return_tensors="pt").to(self.device)
            with torch.no_grad():
                output_ids = self.model.generate(
                    **inputs, max_new_tokens=max_new_tokens, do_sample=True,
                    temperature=0.3, top_p=0.9, pad_token_id=self.tokenizer.eos_token_id,
                )
            generated = output_ids[0][inputs["input_ids"].shape[1]:]
            return self.tokenizer.decode(generated, skip_special_tokens=True).strip()
        except Exception as e:
            return f"Generation failed: {e}"


ai_engine = AIEngine()
print(f"Model status: {'loaded' if ai_engine.is_ready else 'unavailable, ' + str(ai_engine.load_error)}")


## Scoring

Scores are computed from static and structural findings only; the model has no path to influence
a score. Each agent starts at 100 and subtracts a fixed penalty per finding by severity: Critical
30, High 18, Medium 10, Low 4, floored at zero. The overall score is a weighted average: Security
40%, Performance 30%, Code Quality 30%. Penalties are set so a single critical finding is
sufficient to push a domain score below 70.

In [ ]:
SEVERITY_ORDER = {"Critical": 4, "High": 3, "Medium": 2, "Low": 1}
SEVERITY_PENALTY = {"Critical": 30, "High": 18, "Medium": 10, "Low": 4}


def calculate_sub_score(findings) -> int:
    score = 100
    for f in findings:
        score -= SEVERITY_PENALTY.get(f.get("severity", "Low"), 4)
    return max(0, score)


def dedupe_findings(findings):
    seen, unique = set(), []
    for f in findings:
        key = (f.get("line"), f.get("description"))
        if key not in seen:
            seen.add(key)
            unique.append(f)
    return unique


def count_by_severity(findings) -> dict:
    counts = {"Critical": 0, "High": 0, "Medium": 0, "Low": 0}
    for f in findings:
        sev = f.get("severity", "Low")
        if sev in counts:
            counts[sev] += 1
    return counts


def detect_duplicate_lines(code: str):
    """Flags a non-trivial line that repeats three or more times."""
    lines = [l.strip() for l in code.splitlines() if l.strip() and not l.strip().startswith("#")]
    counts = Counter(lines)
    findings = []
    for line, c in counts.items():
        if c >= 3 and len(line) > 20:
            findings.append({
                "line": None, "category": "Code Quality", "severity": "Low",
                "description": f"Line repeated {c} times: `{line[:60]}...`",
                "suggestion": "Extract the repeated logic into a shared function.",
            })
    return findings


## Security Agent: Static Scanner

Regex-based rules covering common `OWASP`-adjacent patterns: SQL injection, command injection,
hardcoded secrets, insecure deserialization, weak hashing, insecure randomness, debug flags left
enabled. This is a pattern scanner, not a data-flow analyzer; it will miss injection built across
multiple statements or routed through helper functions, and it can produce false positives on code
that happens to match a pattern without being exploitable.

The SQL injection rule matches SQL-keyword string concatenation anywhere in the line, not only at
the `execute()` call site, since the vulnerable string is frequently assembled in a variable
first. The hardcoded-secret rule matches the target keyword as a substring of the identifier, so
`ADMIN_PASSWORD` and `DB_SECRET_KEY` are caught in addition to a bare `password` assignment.

In [ ]:
class StaticSecurityScanner:
    def __init__(self):
        sql_kw = r'\b(SELECT|INSERT|UPDATE|DELETE|DROP)\b'
        self.rules = [
            {"category": "SQL Injection", "severity": "Critical",
             "pattern": re.compile(rf'[\'"].*{sql_kw}.*[\'"]\s*\+', re.IGNORECASE),
             "message": "SQL query string built with `+` concatenation on user-controlled input.",
             "suggestion": "Use parameterized queries: cursor.execute(\"... WHERE username = %s\", (username,))."},
            {"category": "SQL Injection", "severity": "Critical",
             "pattern": re.compile(rf'f[\'"].*{sql_kw}.*\{{[^}}]+\}}', re.IGNORECASE),
             "message": "SQL query built as an f-string with an embedded variable.",
             "suggestion": "Use parameterized queries instead of interpolating variables into the SQL string."},
            {"category": "SQL Injection", "severity": "Critical",
             "pattern": re.compile(rf'[\'"].*{sql_kw}.*%s.*[\'"]\s*%', re.IGNORECASE),
             "message": "SQL query built with `%` string formatting.",
             "suggestion": "Pass parameters as a tuple to execute() instead of formatting the query string."},
            {"category": "Command Injection", "severity": "Critical",
             "pattern": re.compile(r'\b(os\.system|subprocess\.(Popen|call|run)\s*\([^)]*shell\s*=\s*True|eval|exec)\s*\('),
             "message": "Dynamic command or code execution on potentially untrusted input.",
             "suggestion": "Avoid shell=True and eval/exec on untrusted input; pass an argument list to subprocess instead."},
            {"category": "Hardcoded Secret", "severity": "High",
             "pattern": re.compile(r'\b\w*(password|passwd|secret|api_?key|token)\w*\s*=\s*[\'"][^\'"]{3,}[\'"]', re.IGNORECASE),
             "message": "Credential or key assigned as a literal in source.",
             "suggestion": "Load secrets from environment variables or a secrets manager."},
            {"category": "Insecure Deserialization", "severity": "High",
             "pattern": re.compile(r'\bpickle\.loads?\s*\(|yaml\.load\s*\((?!.*SafeLoader)'),
             "message": "Deserialization method that can execute arbitrary code on untrusted input.",
             "suggestion": "Use yaml.safe_load(), or avoid pickle for data from an untrusted source."},
            {"category": "Weak Hashing", "severity": "Medium",
             "pattern": re.compile(r'hashlib\.(md5|sha1)\s*\('),
             "message": "MD5 or SHA1 used, unsuitable for password storage.",
             "suggestion": "Use bcrypt, scrypt, or Argon2 for password hashing."},
            {"category": "Insecure Randomness", "severity": "Medium",
             "pattern": re.compile(r'\brandom\.(random|randint|choice)\s*\('),
             "message": "`random` module used for a value that appears security-sensitive.",
             "suggestion": "Use the `secrets` module for tokens or credentials."},
            {"category": "Debug Mode Enabled", "severity": "Low",
             "pattern": re.compile(r'debug\s*=\s*True', re.IGNORECASE),
             "message": "Debug flag enabled; may expose stack traces if reached in production.",
             "suggestion": "Confirm debug mode is disabled before deployment."},
        ]

    def scan(self, code: str):
        findings = []
        for i, line in enumerate(code.splitlines(), start=1):
            for rule in self.rules:
                if rule["pattern"].search(line):
                    findings.append({
                        "line": i, "category": rule["category"], "severity": rule["severity"],
                        "description": rule["message"], "suggestion": rule["suggestion"],
                    })
        findings = dedupe_findings(findings)
        findings.sort(key=lambda f: SEVERITY_ORDER[f["severity"]], reverse=True)
        return findings


## Performance and Quality Agents: Structural Checkers

Regex is insufficient for structural questions such as loop nesting or function length, so these
two checkers walk the AST instead. Both require code that parses as valid Python; a `SyntaxError`
is caught upstream and the agent falls back to model-only analysis for that run.

The performance checker treats a loop nested inside another loop over the *same* iterable as a
distinct, higher-confidence case: this is the pattern that previously led the model to suggest a
list comprehension as a complexity fix. The finding text states directly that only an algorithmic
change (a set or dict lookup) reduces complexity, and that a syntax rewrite does not.

The quality checker flags long functions, missing docstrings, single-character parameter names
outside a small allow-list (`i, j, k, x, y, z, n, m, _`), and bare `except:` clauses. The
allow-list and the 20-character minimum in `detect_duplicate_lines` exist specifically to avoid
flagging idiomatic short names and incidental short-line repeats as defects.

In [ ]:
class PerformanceChecker(ast.NodeVisitor):
    IO_LIKE_CALLS = {"execute", "get", "post", "open", "connect", "read", "write", "commit"}

    def __init__(self):
        self.findings = []
        self.loop_stack = []

    def _add(self, line, message, severity, suggestion):
        self.findings.append({"line": line, "category": "Performance", "severity": severity,
                               "description": message, "suggestion": suggestion})

    @staticmethod
    def _call_name(func_node):
        if isinstance(func_node, ast.Attribute):
            return func_node.attr
        if isinstance(func_node, ast.Name):
            return func_node.id
        return ""

    def _check_loop_body(self, node):
        for child in node.body:
            for sub in ast.walk(child):
                if isinstance(sub, ast.Call):
                    fname = self._call_name(sub.func)
                    if fname in self.IO_LIKE_CALLS:
                        self._add(getattr(sub, "lineno", node.lineno),
                                   f"Call to `{fname}(...)` inside a loop body.", "High",
                                   "Move the call outside the loop if its result does not change per iteration, or batch it.")
                if isinstance(sub, ast.AugAssign) and isinstance(sub.op, ast.Add):
                    self._add(getattr(sub, "lineno", node.lineno),
                               "String or sequence accumulation via `+=` inside a loop.", "Low",
                               "Collect items in a list and join once after the loop; this is a runtime optimization, not a complexity change.")

    def _visit_loop(self, node):
        self.loop_stack.append(node)
        if len(self.loop_stack) >= 2:
            outer = self.loop_stack[-2]
            same_source = (
                isinstance(node, ast.For) and isinstance(outer, ast.For)
                and ast.dump(node.iter) == ast.dump(outer.iter)
            )
            if same_source:
                self._add(node.lineno,
                           "Nested loop iterating twice over the same collection: O(n squared).",
                           "Medium",
                           "Build a set or dict once and use O(1) lookups instead of the nested scan. "
                           "A syntax rewrite such as a list comprehension does not change the complexity.")
            else:
                self._add(node.lineno,
                           "Nested loops detected; likely O(n squared) or worse depending on the loop body.",
                           "Medium",
                           "Check whether the inner loop exists to search or check membership; if so, a set or dict "
                           "removes the need for it. A syntax rewrite alone does not change complexity.")
        self._check_loop_body(node)
        self.generic_visit(node)
        self.loop_stack.pop()

    def visit_For(self, node):
        self._visit_loop(node)

    def visit_While(self, node):
        self._visit_loop(node)


class CodeQualityChecker(ast.NodeVisitor):
    MAX_FUNCTION_LINES = 40
    ACCEPTABLE_SHORT_NAMES = {"i", "j", "k", "x", "y", "z", "n", "m", "_"}

    def __init__(self):
        self.findings = []

    def _add(self, line, message, severity, suggestion):
        self.findings.append({"line": line, "category": "Code Quality", "severity": severity,
                               "description": message, "suggestion": suggestion})

    def visit_FunctionDef(self, node):
        end = getattr(node, "end_lineno", None)
        if end and (end - node.lineno) > self.MAX_FUNCTION_LINES:
            self._add(node.lineno, f"Function `{node.name}` is {end - node.lineno} lines long.", "Medium",
                       "Split into smaller functions, each with a single responsibility.")
        if not ast.get_docstring(node):
            self._add(node.lineno, f"Function `{node.name}` has no docstring.", "Low",
                       "Document inputs, output, and side effects.")
        for arg in node.args.args:
            if len(arg.arg) <= 1 and arg.arg not in self.ACCEPTABLE_SHORT_NAMES:
                self._add(node.lineno, f"Parameter `{arg.arg}` in `{node.name}` is not descriptive.", "Low",
                           "Use a name that conveys the parameter's purpose.")
        self.generic_visit(node)

    def visit_ExceptHandler(self, node):
        if node.type is None:
            self._add(node.lineno, "Bare `except:` clause; masks unrelated exceptions.", "Medium",
                       "Catch a specific exception type instead.")
        self.generic_visit(node)


## Security Agent

Runs `StaticSecurityScanner`, then passes the findings to the model with a prompt restricted to the security domain. The model is expected to explain the listed findings and, only where directly evidenced, add further issues; otherwise it returns the fixed no-finding string.

In [ ]:
class SecurityAgent:
    NAME = "Security Agent"

    def __init__(self, ai_engine: AIEngine):
        self.ai_engine = ai_engine
        self.scanner = StaticSecurityScanner()

    def analyze(self, code: str) -> dict:
        findings = self.scanner.scan(code)
        ai_summary = self.ai_engine.generate(self._build_prompt(code, findings))
        return {"agent": self.NAME, "findings": findings, "score": calculate_sub_score(findings), "ai_summary": ai_summary}

    def _build_prompt(self, code, findings):
        findings_text = "\n".join(
            f"- [{f['severity']}] {f['category']} (line {f['line']}): {f['description']}" for f in findings
        ) or "- No pattern-based issue found by the static scanner."
        return f"""{COMMON_AGENT_RULES}

You are a security reviewer. Analyze the following Python code strictly from a security
standpoint, using the static findings below as your evidence base. For each real issue: state
what it is, its severity, why it matters, and a concrete fix. If nothing significant is present
beyond the listed findings, respond with exactly: "No significant issue detected."

Code:
```python
{code}
```

Static findings:
{findings_text}
"""


## Performance Agent

Runs `PerformanceChecker` against the parsed AST, adds a small set of regex checks for library-specific anti-patterns (`iterrows`), then queries the model under `PERFORMANCE_RULES`, which forbids treating a syntax rewrite as a complexity fix.

In [ ]:
class PerformanceAgent:
    NAME = "Performance Agent"

    def __init__(self, ai_engine: AIEngine):
        self.ai_engine = ai_engine

    def analyze(self, code: str) -> dict:
        findings, note = [], ""
        try:
            tree = ast.parse(code)
            checker = PerformanceChecker()
            checker.visit(tree)
            findings = checker.findings
        except SyntaxError:
            note = "Code did not parse; structural checks skipped, model-only analysis used."
        findings += self._regex_checks(code)
        findings = dedupe_findings(findings)

        ai_summary = self.ai_engine.generate(self._build_prompt(code, findings))
        return {"agent": self.NAME, "findings": findings, "score": calculate_sub_score(findings),
                "ai_summary": ai_summary, "note": note}

    def _regex_checks(self, code):
        findings = []
        for i, line in enumerate(code.splitlines(), start=1):
            if re.search(r'\.iterrows\s*\(\s*\)', line):
                findings.append({"line": i, "category": "Performance", "severity": "Medium",
                                  "description": "`iterrows()` used; slow on large DataFrames.",
                                  "suggestion": "Use `itertuples()` or a vectorized operation."})
        return findings

    def _build_prompt(self, code, findings):
        findings_text = "\n".join(
            f"- [{f['severity']}] line {f.get('line', '?')}: {f['description']} Suggestion: {f.get('suggestion', '')}"
            for f in findings
        ) or "- No structural performance issue found."
        return f"""{COMMON_AGENT_RULES}
{PERFORMANCE_RULES}

You are a performance reviewer. Analyze the following Python code strictly for performance,
using the structural findings below as your evidence base. If nothing significant is present
beyond them, respond with exactly: "No significant issue detected."

Code:
```python
{code}
```

Structural findings:
{findings_text}
"""


## Code Quality Agent

Runs `CodeQualityChecker` and `detect_duplicate_lines`, then queries the model for readability and maintainability feedback only. The prompt explicitly restricts "does too many things" to functions that are long or visibly mix unrelated responsibilities.

In [ ]:
class CodeQualityAgent:
    NAME = "Code Quality Agent"

    def __init__(self, ai_engine: AIEngine):
        self.ai_engine = ai_engine

    def analyze(self, code: str) -> dict:
        findings, note = [], ""
        try:
            tree = ast.parse(code)
            checker = CodeQualityChecker()
            checker.visit(tree)
            findings = checker.findings
        except SyntaxError:
            note = "Code did not parse; structural checks skipped, model-only analysis used."
        findings += detect_duplicate_lines(code)
        findings = dedupe_findings(findings)

        ai_summary = self.ai_engine.generate(self._build_prompt(code, findings))
        return {"agent": self.NAME, "findings": findings, "score": calculate_sub_score(findings),
                "ai_summary": ai_summary, "note": note}

    def _build_prompt(self, code, findings):
        findings_text = "\n".join(
            f"- [{f['severity']}] line {f.get('line', '?')}: {f['description']}" for f in findings
        ) or "- No structural quality issue found."
        return f"""{COMMON_AGENT_RULES}

You are a code reviewer. Analyze the following Python code strictly for readability, naming,
structure, duplication, and error handling, using the findings below as your evidence base. Only
describe a function as doing too many things if it is long or visibly mixes unrelated
responsibilities; do not say this about a short, single-purpose function. If nothing significant
is present beyond the listed findings, respond with exactly: "No significant issue detected."

Code:
```python
{code}
```

Structural findings:
{findings_text}
"""


## Report Agent

Aggregates the three domain results into a weighted score, renders the combined Markdown report, generates a short executive summary from the pooled findings, and, on request, produces a full rewrite of the input addressing the findings.

In [ ]:
class ReportAgent:
    WEIGHTS = {"security": 0.4, "performance": 0.3, "quality": 0.3}

    DISCLAIMER = (
        "Static analysis is heuristic and may miss vulnerabilities or produce false positives. "
        "Model-generated explanations may contain mistakes. Performance findings are estimates "
        "unless verified through benchmarking. Review and test any suggested fix before applying it."
    )

    def build(self, security_result, performance_result, quality_result) -> dict:
        overall = round(
            security_result["score"] * self.WEIGHTS["security"]
            + performance_result["score"] * self.WEIGHTS["performance"]
            + quality_result["score"] * self.WEIGHTS["quality"], 1,
        )
        return {
            "security": security_result, "performance": performance_result, "quality": quality_result,
            "overall_score": overall, "timestamp": datetime.datetime.now().isoformat(timespec="seconds"),
            "language": "Python",
        }

    def _findings_table(self, findings):
        if not findings:
            return "No significant issue detected.\n"
        rows = ["| Severity | Line | Finding | Recommendation |", "|---|---|---|---|"]
        for f in findings:
            rows.append(f"| {f.get('severity', '-')} | {f.get('line', '-') or '-'} | "
                        f"{f.get('description', '-')} | {f.get('suggestion', '-') or '-'} |")
        return "\n".join(rows)

    def generate_executive_summary(self, ai_engine: AIEngine, report: dict) -> str:
        all_findings = report["security"]["findings"] + report["performance"]["findings"] + report["quality"]["findings"]
        if not all_findings:
            return "No significant issues detected across the three agents."
        findings_text = "\n".join(
            f"- [{f.get('severity', '-')}] ({f.get('category', '-')}) {f.get('description', '')}" for f in all_findings
        )
        prompt = f"""{COMMON_AGENT_RULES}

Write a short executive summary of a code review using only the findings below. In 3 to 5
sentences, order the most important issues by severity. Do not restate every finding.

Findings:
{findings_text}
"""
        return ai_engine.generate(prompt, max_new_tokens=250)

    def to_markdown(self, report: dict) -> str:
        sec, perf, qual = report["security"], report["performance"], report["quality"]
        all_findings = sec["findings"] + perf["findings"] + qual["findings"]
        counts = count_by_severity(all_findings)

        md = [
            f"# {PROJECT_NAME} Report",
            f"Generated: {report['timestamp']}",
            f"Language: {report.get('language', 'Python')}",
            f"\n## Overall score: {report['overall_score']} / 100\n",
            "Each agent starts at 100 and subtracts a fixed penalty per finding: Critical "
            f"-{SEVERITY_PENALTY['Critical']}, High -{SEVERITY_PENALTY['High']}, "
            f"Medium -{SEVERITY_PENALTY['Medium']}, Low -{SEVERITY_PENALTY['Low']}, floored at 0. "
            "Overall is Security 40%, Performance 30%, Code Quality 30%. This calculation runs in "
            "Python and does not use the model.\n",
            "| Agent | Score |", "|---|---|",
            f"| Security | {sec['score']} / 100 |",
            f"| Performance | {perf['score']} / 100 |",
            f"| Code Quality | {qual['score']} / 100 |",
            f"\nFindings by severity: Critical {counts['Critical']}, High {counts['High']}, "
            f"Medium {counts['Medium']}, Low {counts['Low']}",
        ]

        if report.get("executive_summary"):
            md.append("\n## Executive summary\n" + report["executive_summary"])

        for label, result in (("Security Agent", sec), ("Performance Agent", perf), ("Code Quality Agent", qual)):
            md.append(f"\n---\n## {label}")
            if result.get("note"):
                md.append(f"Note: {result['note']}")
            md.append(self._findings_table(result["findings"]))
            md.append("\nAnalysis:\n" + result["ai_summary"])

        if report.get("improved_code"):
            md.append("\n---\n## Suggested patch (model-generated)")
            md.append("Review and test before applying.\n")
            md.append(f"```python\n{report['improved_code']}\n```")

        md.append("\n---\n" + self.DISCLAIMER)
        return "\n".join(md)

    def generate_improved_code(self, ai_engine: AIEngine, code: str, report: dict) -> str:
        all_findings = report["security"]["findings"] + report["performance"]["findings"] + report["quality"]["findings"]
        findings_text = "\n".join(f"- [{f.get('severity', '-')}] {f.get('description', '')}" for f in all_findings) or "- No major findings."
        prompt = f"""{COMMON_AGENT_RULES}

Rewrite the Python code below to address the findings listed, preserving the original
functionality. Return a single code block.

Original code:
```python
{code}
```

Findings:
{findings_text}
"""
        return ai_engine.generate(prompt, max_new_tokens=900)


## Orchestrator

Validates input, runs the three domain agents in sequence, and builds the final report. Rejects empty, too-short, or excessively long input before invoking any agent. Supports Python only; no other language is listed in the interface, since none of the structural checks apply to it.

In [ ]:
class CodeReviewOrchestrator:
    def __init__(self, ai_engine: AIEngine):
        self.ai_engine = ai_engine
        self.security_agent = SecurityAgent(ai_engine)
        self.performance_agent = PerformanceAgent(ai_engine)
        self.quality_agent = CodeQualityAgent(ai_engine)
        self.report_agent = ReportAgent()

    def run(self, code: str) -> dict:
        code = (code or "").strip()
        if not code:
            return {"error": "No code provided."}
        if len(code.splitlines()) < 2:
            return {"error": "Input is too short for a meaningful analysis."}
        if len(code) > 20000:
            return {"error": "Input exceeds the 20,000 character limit for this demo."}

        security_result = self.security_agent.analyze(code)
        performance_result = self.performance_agent.analyze(code)
        quality_result = self.quality_agent.analyze(code)

        report = self.report_agent.build(security_result, performance_result, quality_result)
        report["original_code"] = code
        report["executive_summary"] = self.report_agent.generate_executive_summary(self.ai_engine, report)
        return report

    def generate_improved_code(self, code: str, report: dict) -> str:
        return self.report_agent.generate_improved_code(self.ai_engine, code, report)


orchestrator = CodeReviewOrchestrator(ai_engine)


## Sample Input

Contains SQL injection via a variable (not inline in the `execute()` call), a hardcoded credential using a compound identifier, MD5 hashing, a nested loop over a shared collection, and a bare `except`. Used to verify each fix listed in the changelog and to exercise all three agents in one pass.

In [ ]:
demo_code = """
import sqlite3
import hashlib

ADMIN_PASSWORD = "admin123"

def login(username, password):
    try:
        conn = sqlite3.connect("users.db")
        cursor = conn.cursor()
        query = "SELECT * FROM users WHERE username = '" + username + "' AND password = '" + password + "'"
        cursor.execute(query)
        user = cursor.fetchone()
    except:
        return False

    hashed_password = hashlib.md5(password.encode()).hexdigest()

    all_users = cursor.execute("SELECT username FROM users").fetchall()
    matches = []
    for a in all_users:
        for b in all_users:
            if a == b:
                matches.append(a)

    if user:
        return True
    return False
"""


## Pipeline Test

Runs the sample input through the orchestrator and checks that the SQL injection finding is present, confirming the scanner fix.

In [ ]:
report = orchestrator.run(demo_code)

if "error" in report:
    print(report["error"])
else:
    sql_findings = [f for f in report["security"]["findings"] if f["category"] == "SQL Injection"]
    print(f"SQL Injection findings: {len(sql_findings)}")
    display(Markdown(orchestrator.report_agent.to_markdown(report)))


## Patch Generation

Optional. Output is model-generated and unverified; it is not run or tested by this notebook and should be reviewed before use.

In [ ]:
if "error" not in report:
    improved_code = orchestrator.generate_improved_code(report["original_code"], report)
    report["improved_code"] = improved_code
    display(Markdown(
        "### Suggested patch (model-generated)\n\n"
        f"```python\n{improved_code}\n```\n\n"
        "Review and test before applying."
    ))
else:
    print("No report available.")


## Interface Components

HTML fragments for the header, the pipeline indicator, and the score dashboard. Kept separate from the Blocks definition below to keep that cell focused on layout and event wiring.

In [ ]:
CG_FLOW_HTML = ('<div class="cg-flow">Source &rarr; Security Agent &rarr; Performance Agent '
                '&rarr; Code Quality Agent &rarr; Report Agent &rarr; Recommendations</div>')


def build_header_html(ai_ready: bool) -> str:
    status_class = "ready" if ai_ready else "off"
    status_text = "Model ready" if ai_ready else "Model unavailable, static analysis only"
    return f"""<div class="cg-header">
        {LOGO_SVG}
        <div>
            <div class="cg-title">{PROJECT_NAME}</div>
            <div class="cg-subtitle">Static and AI-assisted analysis for security, performance, and maintainability</div>
        </div>
        <div class="cg-status"><span class="cg-dot {status_class}"></span>{status_text}</div>
    </div>"""


def build_score_cards_html(report: dict) -> str:
    sec, perf, qual = report["security"]["score"], report["performance"]["score"], report["quality"]["score"]
    all_findings = report["security"]["findings"] + report["performance"]["findings"] + report["quality"]["findings"]
    counts = count_by_severity(all_findings)
    return f"""<div>
      <div class="cg-overall"><span class="label">Overall</span><span class="value">{report['overall_score']} / 100</span></div>
      <div class="cg-cards">
        <div class="cg-card security"><div class="label">Security</div><div class="value">{sec} / 100</div></div>
        <div class="cg-card performance"><div class="label">Performance</div><div class="value">{perf} / 100</div></div>
        <div class="cg-card quality"><div class="label">Quality</div><div class="value">{qual} / 100</div></div>
      </div>
      <div class="cg-issues">Critical {counts['Critical']} &nbsp; High {counts['High']} &nbsp; Medium {counts['Medium']} &nbsp; Low {counts['Low']}</div>
    </div>"""


## Interface

Layout: source input on the left, score dashboard and findings on the right. `Run Analysis` triggers the orchestrator; `Suggest Fix` and the report download only become available once a report exists.

In [ ]:
import gradio as gr


def analyze_ui(code):
    report = orchestrator.run(code)
    if "error" in report:
        return "", report["error"], None, gr.update(visible=False), gr.update(visible=False)

    dashboard_html = build_score_cards_html(report)
    detail_md = orchestrator.report_agent.to_markdown(report)

    report_path = "/content/codeguard_report.md"
    try:
        with open(report_path, "w", encoding="utf-8") as f:
            f.write(detail_md)
        file_update = gr.update(value=report_path, visible=True)
    except Exception:
        file_update = gr.update(visible=False)

    return dashboard_html, detail_md, report, gr.update(visible=True), file_update


def improve_ui(report_state):
    if not report_state:
        return "Run analysis first."
    improved = orchestrator.generate_improved_code(report_state["original_code"], report_state)
    return (
        "### Suggested patch (model-generated)\n\n"
        f"```python\n{improved}\n```\n\n"
        "Review and test before applying."
    )


with gr.Blocks(title=PROJECT_NAME, css=CUSTOM_CSS) as demo:
    gr.HTML(build_header_html(ai_engine.is_ready))
    gr.HTML(CG_FLOW_HTML)

    with gr.Row():
        with gr.Column(scale=1):
            code_input = gr.Code(label="Source", language="python", lines=18)
            lang_dropdown = gr.Dropdown(choices=["Python"], value="Python", label="Language",
                                         interactive=False, info="Python only")
            with gr.Row():
                demo_btn = gr.Button("Load Sample")
                analyze_btn = gr.Button("Run Analysis", variant="primary")
            improve_btn = gr.Button("Suggest Fix", visible=False)
            report_file = gr.File(label="Report", visible=False)
        with gr.Column(scale=1):
            dashboard_output = gr.HTML()
            report_output = gr.Markdown()

    improved_output = gr.Markdown()
    report_state = gr.State(None)

    demo_btn.click(fn=lambda: demo_code, outputs=[code_input])
    analyze_btn.click(
        fn=analyze_ui,
        inputs=[code_input],
        outputs=[dashboard_output, report_output, report_state, improve_btn, report_file],
    )
    improve_btn.click(fn=improve_ui, inputs=[report_state], outputs=[improved_output])


## Launch

In [ ]:
demo.launch(share=True, debug=False)


## Limitations

Static rules are pattern-based and operate per line or per AST node; they do not track data flow
across statements or function boundaries, so injection or secret exposure spread across multiple
lines can be missed. Structural checks (loop nesting, function length, docstring presence) apply
to Python only and are skipped when the input does not parse.

Model output is generated conditioned on the static findings but is not otherwise constrained; it
can still contain inaccurate explanations or, in rare cases, extend beyond the evidence despite the
prompt constraints. The scoring formula is deterministic and independent of the model, but the
severities assigned by the static rules and the AST checkers are fixed heuristics, not a formal
risk assessment.

Performance findings describe complexity or call patterns observed in the source; they are not
measurements. Claims about runtime require benchmarking on representative input.

The suggested patch is generated in a single pass and is not executed or tested by this notebook.
It should be reviewed line by line before being applied to a codebase.
